In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
PROCESSED_DIR = "../processed"
file_path = os.path.join(PROCESSED_DIR, "merged_daily.csv")
df = pd.read_csv(file_path)


In [3]:

print("First 5 rows:\n", df.head())
print("\nColumns:", df.columns.tolist())

First 5 rows:
                         date  tavg  tmin  tmax  prcp  wspd    pres   pm10  \
0  2023-01-01 00:00:00+07:00  26.5  23.0  31.0   0.6   8.9  1012.7  42.38   
1  2023-01-02 00:00:00+07:00  26.7  23.0  32.0   0.0   6.3  1013.2  51.38   
2  2023-01-03 00:00:00+07:00  28.1  24.0  33.0   1.0   9.7  1012.4  40.10   
3  2023-01-04 00:00:00+07:00  27.6  24.0  33.0   0.1  13.4  1011.7  28.11   
4  2023-01-05 00:00:00+07:00  25.8  24.0  28.0   5.8  17.2  1011.3  16.40   

   pm2_5  nitrogen_dioxide  sulphur_dioxide  ozone  carbon_monoxide  \
0  29.00             42.76            17.00  51.33           710.33   
1  35.54             53.81             9.94  57.96          1090.38   
2  27.82             39.09             5.94  54.12           662.62   
3  19.46             28.41            13.16  46.79           495.75   
4  11.28             21.09            11.22  51.29           352.62   

   uv_index_max  uv_index_clear_sky_max  
0          7.20                    8.80  
1          

In [4]:
# Count days exceeding WHO 24h PM2.5 limit (>15 µg/m³)
df['pm25_exceed_who15'] = df['pm2_5'] > 15
exceed_days = int(df['pm25_exceed_who15'].sum())
total_days = len(df)
percent_exceed = exceed_days / total_days * 100
print(f"Days exceeding WHO 24h PM2.5 limit (>15 µg/m³): {exceed_days} / {total_days} ({percent_exceed:.1f}%)")

Days exceeding WHO 24h PM2.5 limit (>15 µg/m³): 322 / 365 (88.2%)


In [5]:
# Descriptive statistics for all numeric variables (mean, std, min, max)
numeric_cols = df.select_dtypes(include='number')
summary_stats = numeric_cols.describe().loc[['mean', 'std', 'min', 'max']].T
summary_stats.rename(columns={'mean': 'Mean', 'std': 'Std', 'min': 'Min', 'max': 'Max'}, inplace=True)
summary_stats.round(2)

,Mean,Std,Min,Max
tavg,28.68,1.26,25.10,32.50
tmin,25.33,1.52,21.00,29.00
tmax,33.33,1.68,27.00,38.00
prcp,6.04,7.96,0.00,56.10
wspd,10.33,3.61,3.90,21.50
pres,1009.64,1.93,1004.40,1015.30
pm10,35.80,12.53,13.07,79.53
pm2_5,24.13,8.55,8.97,54.45
nitrogen_dioxide,28.93,9.76,6.89,70.14
sulphur_dioxide,20.69,12.57,1.72,56.39


In [6]:
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')
# Weekly aggregation: weather and air-quality indicators
weekly = df.resample("W").agg({
    # Temperature (°C)
    'tavg': ['mean', lambda x: x.quantile(0.5), lambda x: x.quantile(0.95)],
    'tmin': 'min',
    'tmax': 'max',
    
    # Rainfall (mm)
    'prcp': ['sum', lambda x: (x >= 1).sum()],
    
    # Wind (km/h)
    'wspd': 'mean',
    
    # Pressure (hPa)
    'pres': ['mean', 'std'],
    
    # Air quality
    'pm2_5': ['mean', lambda x: x.quantile(0.95), lambda x: (x > 25).sum()],
    'pm10': ['mean', lambda x: x.quantile(0.95), lambda x: (x > 50).sum()],
    'uv_index_max': 'mean',
    'uv_index_clear_sky_max': 'max',
}).round(2)

weekly.columns = [
    'temp_mean', 'temp_p50', 'temp_p95', 'temp_min', 'temp_max',
    'rain_total_mm', 'rain_days_ge1mm',
    'wind_mean_kmh',
    'pressure_mean_hPa', 'pressure_std_hPa',
    'pm25_mean', 'pm25_p95', 'pm25_days_exceed',
    'pm10_mean', 'pm10_p95', 'pm10_days_exceed',
    'uv_mean', 'uv_max_daily',
]

output_file = os.path.join(PROCESSED_DIR, "data_weekly.csv")
weekly.to_csv(output_file, index=True)


In [7]:
weekly.head(10)


,temp_mean,temp_p50,temp_p95,temp_min,temp_max,rain_total_mm,rain_days_ge1mm,wind_mean_kmh,pressure_mean_hPa,pressure_std_hPa,pm25_mean,pm25_p95,pm25_days_exceed,pm10_mean,pm10_p95,pm10_days_exceed,uv_mean,uv_max_daily
date,,,,,,,,,,,,,,,,,,
2023-01-01 00:00:00+07:00,26.50,26.5,26.50,23.0,31.0,0.6,0,8.90,1012.70,NaN,29.00,29.00,1,42.38,42.38,0,7.20,8.80
2023-01-08 00:00:00+07:00,26.53,25.9,27.95,22.0,33.0,31.4,5,11.71,1011.50,1.02,20.66,33.22,3,29.82,48.00,1,5.91,10.05
2023-01-15 00:00:00+07:00,27.56,27.5,28.67,24.0,33.0,49.0,4,9.34,1010.33,1.21,28.22,51.05,3,41.05,73.20,3,7.26,9.80
2023-01-22 00:00:00+07:00,28.00,27.6,28.87,23.0,34.0,0.4,0,7.43,1010.84,1.09,38.73,47.87,7,56.44,69.41,5,7.76,9.35
2023-01-29 00:00:00+07:00,27.33,27.2,28.84,23.0,35.0,2.1,1,8.46,1009.57,0.63,32.34,47.69,5,47.43,69.72,3,8.59,10.45
2023-02-05 00:00:00+07:00,27.79,27.8,29.34,21.0,35.0,2.9,1,9.36,1009.97,0.85,27.97,48.94,3,41.53,72.70,3,8.94,11.25
2023-02-12 00:00:00+07:00,29.04,29.0,29.41,25.0,35.0,1.5,0,12.31,1009.89,0.41,14.98,17.49,0,22.65,26.20,0,9.73,11.70
2023-02-19 00:00:00+07:00,28.14,28.0,28.57,23.0,34.0,4.0,1,8.74,1010.99,0.41,22.29,30.76,3,33.17,44.93,0,9.28,11.80
2023-02-26 00:00:00+07:00,27.63,27.6,27.87,22.0,34.0,0.9,0,7.73,1011.97,1.43,27.97,32.84,6,41.90,48.78,0,9.82,12.45


In [8]:
# Monthly aggregation with the same indicator set
monthly = df.resample("ME").agg({
    'tavg': ['mean', lambda x: x.quantile(0.5), lambda x: x.quantile(0.95)],
    'tmin': 'min',
    'tmax': 'max',
    'prcp': ['sum', lambda x: (x >= 1).sum()],
    'wspd': 'mean',
    'pres': ['mean', 'std'],
    'pm2_5': ['mean', lambda x: x.quantile(0.95), lambda x: (x > 25).sum()],
    'pm10': ['mean', lambda x: x.quantile(0.95), lambda x: (x > 50).sum()],
    'uv_index_max': 'mean',
    'uv_index_clear_sky_max': 'max',
}).round(2)
monthly.columns = [
    'temp_mean', 'temp_p50', 'temp_p95', 'temp_min', 'temp_max',
    'rain_total_mm', 'rain_days_ge1mm',
    'wind_mean_kmh',
    'pressure_mean_hPa', 'pressure_std_hPa',
    'pm25_mean', 'pm25_p95', 'pm25_days_exceed',
    'pm10_mean', 'pm10_p95', 'pm10_days_exceed',
    'uv_mean', 'uv_max_daily',
]


In [9]:
monthly.head(10)

,temp_mean,temp_p50,temp_p95,temp_min,temp_max,rain_total_mm,rain_days_ge1mm,wind_mean_kmh,pressure_mean_hPa,pressure_std_hPa,pm25_mean,pm25_p95,pm25_days_exceed,pm10_mean,pm10_p95,pm10_days_exceed,uv_mean,uv_max_daily
date,,,,,,,,,,,,,,,,,,
2023-01-31 00:00:00+07:00,27.25,27.20,28.85,21.0,35.0,83.5,10,9.17,1010.65,1.21,30.91,51.49,21,45.14,75.72,14,7.44,10.45
2023-02-28 00:00:00+07:00,28.30,28.25,29.33,22.0,35.0,9.3,2,9.52,1010.92,1.46,21.72,33.17,10,32.41,48.86,1,9.72,12.95
2023-03-31 00:00:00+07:00,28.44,28.60,29.90,21.0,36.0,3.1,2,10.73,1011.34,1.97,19.12,25.70,5,29.35,39.88,0,10.50,13.60
2023-04-30 00:00:00+07:00,30.50,30.60,31.30,23.0,37.0,58.7,10,11.22,1008.64,1.41,24.10,41.71,13,36.53,61.97,7,8.26,12.95
2023-05-31 00:00:00+07:00,29.98,30.10,31.95,23.0,38.0,233.7,24,10.94,1008.45,1.61,24.59,35.62,16,36.58,52.39,3,8.09,11.85
2023-06-30 00:00:00+07:00,29.14,28.70,30.70,23.0,36.0,353.1,29,11.61,1007.95,0.83,28.39,42.60,19,41.69,62.82,8,8.47,11.50
2023-07-31 00:00:00+07:00,28.12,28.00,29.90,23.0,35.0,422.6,30,12.17,1007.96,1.56,19.95,30.49,8,29.25,44.81,0,7.93,12.50
2023-08-31 00:00:00+07:00,29.20,29.10,30.25,24.0,35.0,237.1,27,15.03,1008.39,1.21,19.61,26.29,2,28.75,38.29,0,9.74,12.65
2023-09-30 00:00:00+07:00,27.98,27.80,29.10,24.0,35.0,334.4,29,11.14,1008.52,1.49,21.52,34.04,8,31.44,49.66,1,8.53,13.35


In [10]:
# Save monthly indicators
output_file = os.path.join(PROCESSED_DIR, "data_monthly.csv")
monthly.to_csv(output_file, index=True)

In [11]:
# Create indexed version of monthly indicators (first month = 100)
baseline = monthly.iloc[0]
monthly_indexed = (monthly / baseline * 100).round(2)

output_indexed = os.path.join(PROCESSED_DIR, "data_monthly_indexed.csv")
monthly_indexed.to_csv(output_indexed, index=True)
print(f"Saved indexed monthly data to {output_indexed}")

Saved indexed monthly data to ../processed\data_monthly_indexed.csv


In [12]:
monthly_indexed.head(10)

,temp_mean,temp_p50,temp_p95,temp_min,temp_max,rain_total_mm,rain_days_ge1mm,wind_mean_kmh,pressure_mean_hPa,pressure_std_hPa,pm25_mean,pm25_p95,pm25_days_exceed,pm10_mean,pm10_p95,pm10_days_exceed,uv_mean,uv_max_daily
date,,,,,,,,,,,,,,,,,,
2023-01-31 00:00:00+07:00,100.00,100.00,100.00,100.00,100.00,100.00,100.0,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2023-02-28 00:00:00+07:00,103.85,103.86,101.66,104.76,100.00,11.14,20.0,103.82,100.03,120.66,70.27,64.42,47.62,71.80,64.53,7.14,130.65,123.92
2023-03-31 00:00:00+07:00,104.37,105.15,103.64,100.00,102.86,3.71,20.0,117.01,100.07,162.81,61.86,49.91,23.81,65.02,52.67,0.00,141.13,130.14
2023-04-30 00:00:00+07:00,111.93,112.50,108.49,109.52,105.71,70.30,100.0,122.36,99.80,116.53,77.97,81.01,61.90,80.93,81.84,50.00,111.02,123.92
2023-05-31 00:00:00+07:00,110.02,110.66,110.75,109.52,108.57,279.88,240.0,119.30,99.78,133.06,79.55,69.18,76.19,81.04,69.19,21.43,108.74,113.40
2023-06-30 00:00:00+07:00,106.94,105.51,106.41,109.52,102.86,422.87,290.0,126.61,99.73,68.60,91.85,82.73,90.48,92.36,82.96,57.14,113.84,110.05
2023-07-31 00:00:00+07:00,103.19,102.94,103.64,109.52,100.00,506.11,300.0,132.72,99.73,128.93,64.54,59.22,38.10,64.80,59.18,0.00,106.59,119.62
2023-08-31 00:00:00+07:00,107.16,106.99,104.85,114.29,100.00,283.95,270.0,163.90,99.78,100.00,63.44,51.06,9.52,63.69,50.57,0.00,130.91,121.05
2023-09-30 00:00:00+07:00,102.68,102.21,100.87,114.29,100.00,400.48,290.0,121.48,99.79,123.14,69.62,66.11,38.10,69.65,65.58,7.14,114.65,127.75


In [13]:
# Documentation table for all derived indicators
indicators_doc = pd.DataFrame({
    'Indicator': [
        'temp_mean', 'temp_p50', 'temp_p95', 'temp_min', 'temp_max',
        'rain_total_mm', 'rain_days_ge1mm',
        'wind_mean_kmh',
        'pressure_mean_hPa', 'pressure_std_hPa',
        'pm25_mean', 'pm25_p95', 'pm25_days_exceed',
        'pm10_mean', 'pm10_p95', 'pm10_days_exceed',
        'uv_mean', 'uv_max_daily',
    ],
    'Purpose': [
        'Average daily temperature', 'Median (p50) of daily temperature', '95th percentile of daily temperature',
        'Minimum daily temperature', 'Maximum daily temperature',
        'Total rainfall in the period', 'Number of rainy days ≥1mm',
        'Average wind speed',
        'Average pressure', 'Standard deviation of pressure',
        'Mean of daily PM2.5', '95th percentile of daily PM2.5', 'Number of days with PM2.5 above 25 µg/m³',
        'Mean of daily PM10', '95th percentile of daily PM10', 'Number of days with PM10 above 50 µg/m³',
        'Average daily UV index', 'Maximum daily clear-sky UV index',
    ],
    'Formula': [
        'mean(tavg)', 'quantile(tavg, 0.5)', 'quantile(tavg, 0.95)',
        'min(tmin)', 'max(tmax)',
        'sum(prcp)', 'count(prcp ≥ 1)',
        'mean(wspd)',
        'mean(pres)', 'std(pres)',
        'mean(pm2_5)', 'quantile(pm2_5, 0.95)', 'count(pm2_5 > 25)',
        'mean(pm10)', 'quantile(pm10, 0.95)', 'count(pm10 > 50)',
        'mean(uv_index_max)', 'max(uv_index_clear_sky_max)',
    ],
    'Unit': [
        '°C', '°C', '°C', '°C', '°C',
        'mm', 'days',
        'km/h',
        'hPa', 'hPa',
        'µg/m³', 'µg/m³', 'days',
        'µg/m³', 'µg/m³', 'days',
        'index', 'index',
    ],
    'Threshold': [
        'N/A', 'N/A', '>32°C (extreme heat)', '<10°C (cold)', '>40°C (danger)',
        '>100mm (heavy rain)', '>10 days (wet month)',
        '>50 km/h (strong wind)',
        '<1000 hPa (low pressure)', '>5 hPa (unstable)',
        '>15 µg/m³ (WHO annual)', '>25 µg/m³ (WHO 24h limit)', '>7 days/month (poor AQI)',
        '>45 µg/m³ (WHO annual)', '>50 µg/m³ (WHO 24h limit)', '>7 days/month (poor AQI)',
        '>6 (high UV)', '>11 (extreme UV)',
    ],
})

doc_path = os.path.join(PROCESSED_DIR, "indicators_documentation.csv")
indicators_doc.to_csv(doc_path, index=False)
print(f"Saved indicators documentation to {doc_path}")
indicators_doc


Saved indicators documentation to ../processed\indicators_documentation.csv


,Indicator,Purpose,Formula,Unit,Threshold
0,temp_mean,Average daily temperature,mean(tavg),°C,N/A
1,temp_p50,Median (p50) of daily temperature,"quantile(tavg, 0.5)",°C,N/A
2,temp_p95,95th percentile of daily temperature,"quantile(tavg, 0.95)",°C,>32°C (extreme heat)
3,temp_min,Minimum daily temperature,min(tmin),°C,<10°C (cold)
4,temp_max,Maximum daily temperature,max(tmax),°C,>40°C (danger)
5,rain_total_mm,Total rainfall in the period,sum(prcp),mm,>100mm (heavy rain)
6,rain_days_ge1mm,Number of rainy days ≥1mm,count(prcp ≥ 1),days,>10 days (wet month)
7,wind_mean_kmh,Average wind speed,mean(wspd),km/h,>50 km/h (strong wind)
8,pressure_mean_hPa,Average pressure,mean(pres),hPa,<1000 hPa (low pressure)
9,pressure_std_hPa,Standard deviation of pressure,std(pres),hPa,>5 hPa (unstable)
